In [0]:
%run "/Workspace/utilities"

In [0]:
# COMMAND ----------

dbutils.widgets.text("load_date", "")
dbutils.widgets.dropdown("load_type", "INCREMENTAL", ["INCREMENTAL", "FULL"])

load_date = dbutils.widgets.get("load_date")
load_type = dbutils.widgets.get("load_type")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Configuration

# COMMAND ----------

# Use config already initialized in utilities
logger = DataLogger("transform_orders")

source_path = config.get_source_path("orders.csv")
silver_path = config.get_silver_path("orders")
customer_silver_path = config.get_silver_path("customers")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Read Data

# COMMAND ----------

# Read CSV from source folder
df_source = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(source_path)

logger.info(f"Read {df_source.count()} orders from source CSV")
display(df_source.limit(5))

# COMMAND ----------

# MAGIC %md
# MAGIC ## Data Transformation

# COMMAND ----------

from pyspark.sql.functions import *
from pyspark.sql.types import *

# Clean and transform orders data
df_transformed = df_source \
    .withColumn("order_date", to_timestamp(col("order_date"))) \
    .withColumn("shipped_date", to_timestamp(col("shipped_date"))) \
    .withColumn("order_status", upper(trim(col("order_status")))) \
    .withColumn("total_amount", col("total_amount").cast("decimal(10,2)")) \
    .withColumn("order_year", year(col("order_date"))) \
    .withColumn("order_month", month(col("order_date"))) \
    .withColumn("order_day", dayofmonth(col("order_date"))) \
    .withColumn(
        "order_value_category",
        when(col("total_amount") < 50, "Low")
        .when((col("total_amount") >= 50) & (col("total_amount") < 200), "Medium")
        .when((col("total_amount") >= 200) & (col("total_amount") < 500), "High")
        .otherwise("Premium")
    ) \
    .filter(col("order_date").isNotNull())

# Remove duplicates
df_deduped = df_transformed.dropDuplicates(["order_id"])

# Add audit columns
df_final = df_deduped \
    .withColumn("created_date", current_timestamp()) \
    .withColumn("modified_date", current_timestamp())

logger.info(f"Transformation completed. Final records: {df_final.count()}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Write to Silver Layer

# COMMAND ----------

try:
    df_final.write \
        .format("delta") \
        .mode("overwrite") \
        .partitionBy("order_year", "order_month") \
        .save(silver_path)
    
    logger.info(f"✅ Successfully wrote {df_final.count()} records to Silver layer")
    
except Exception as e:
    logger.error(f"Failed to write to Silver layer", e)
    raise

# COMMAND ----------

# MAGIC %md
# MAGIC ## Summary Statistics

# COMMAND ----------

print(f"\n📊 Transformation Summary:")
print(f"Source records: {df_source.count()}")
print(f"Final records: {df_final.count()}")

print("\nOrders by Status:")
display(df_final.groupBy("order_status").count().orderBy("count", ascending=False))

print("\nOrders by Value Category:")
display(df_final.groupBy("order_value_category").agg(
    count("*").alias("order_count"),
    sum("total_amount").alias("total_revenue")
).orderBy("total_revenue", ascending=False))